In [13]:
#multiagent
#分为Subagents、Handoffs、Skills和Router

#实例
# 开发一个婚礼策划智能体，它包含以下三个核心功能：
# - 旅行规划：负责为你和宾客前往婚礼目的地寻找合适的机票，制定旅行计划
# - 场地规划：负责根据宾客人数在目的地寻找合适的婚礼场地
# - 音乐规划：负责根据用户需求筛选合适的婚礼歌单，并计算出预算

import sys

if sys.platform == "win32":
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__

from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_community.utilities import SQLDatabase
from langchain.agents import AgentState
from langchain_tavily import TavilySearch
from langchain.tools import tool
from langchain.tools import ToolRuntime
from langchain.messages import HumanMessage, ToolMessage, AIMessage
from langgraph.types import Command
from langchain.agents import create_agent
from dotenv import load_dotenv

load_dotenv()

True

In [14]:
#travel agent
# MCP客户端，包含Time、Kiwi两个MCP

client = MultiServerMCPClient(
        {
             "travel_server": {
                "transport": "http",
                "url": "https://mcp.kiwi.com"
        },
        "time": {
            "transport": "stdio",
            "command": "uvx",
            "args": [
                "mcp-server-time",
                "--local-timezone=Asia/Shanghai"
            ]
        }
        }
)

tools = await client.get_tools()

#tarvel  agent
travel_agent = create_agent(
    model="deepseek-flash",
    tools=tools,
    system_prompt="""
    你是一个旅行助手Agent。你为用户搜索到达婚礼地点的理想航班，您不能再追问任何后续问题。
    您必须根据以下标准找到最佳航班选择：
    - 价格（最低，经济舱）
    - 持续时间（最短）
    - 日期（您认为在该地点举行婚礼的最佳时间）
    为了使事情变得简单，只寻找一张票，单程。
    你不会得到额外的信息，只有出发地和目的地。你的工作是批判性地思考最佳选择。
    一旦你找到了最好的选项，让用户知道你的候选列表。
    调用搜索航班工具时，如果是地点在中国，需要使用zh-cn作为Locale参数。
    """
)

In [23]:
#venue agent

# 定义Tavily web_search工具
web_search = TavilySearch(
    max_results=5,
    topic="general"
)

#创建agent
venue_agent = create_agent(
    model="deepseek-flash",
    tools=[web_search],
    system_prompt="""你是场地专家，你在指定的位置搜索满足要求的场地。
    你不能再追问任何后续问题，你必须根据以下标准找到最佳的场地选择：
    - 价格（最低）
    - 容量（完全匹配）
    - 评价（最高）
    您可能需要进行多次搜索以迭代地找到最佳选项。
    """
)

In [24]:
#playist agent
db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")

@tool
def query_playlist(query: str) -> str:
    """查询数据库中的播放列表"""
    try:
        return db.run(query)
    except Exception as e:
        return f"查询数据库时出错: {str(e)}"

# 测试，查询歌单数据
query_playlist.invoke({"query":"SELECT * FROM Playlist"})

playlist_agent = create_agent(
    model="deepseek-flash",
    tools=[query_playlist],
    system_prompt="""你是播放列表专家。查询sql数据库并为给定类型的婚礼策划完美的播放列表。
    一旦你有了你的播放列表，计算播放列表的总时长和成本，每首歌都有一个相关的价格。
    如果在查询数据库时遇到错误，请尝试通过更改查询来修复它们。
    不要空手而归，继续尝试查询数据库，直到你找到一个歌曲列表。
    您可能需要进行多个查询以迭代地找到最佳选项。
    """
)


In [25]:
#主agent

#定义state

class WeddingState(AgentState):
    origin: str #起点
    destination: str #终点
    guest_count: int #嘉宾人数
    genre: str #婚礼类型

#tools
@tool
def update_state(origin: str, destination: str, guest_count: int, genre: str, 
                 runtime: ToolRuntime) -> Command:
        """当你询问用户，知道下列所有值时更新State：origin、destination、guest_count、genre"""
        return Command(update={
                "origin": origin,
                "destination": destination,
                "guest_count": guest_count,
                "genre": genre,
                "messages":[ToolMessage("成功更新state",
                            tool_call_id = runtime.tool_call_id)]
        })

In [26]:
# 接下来，按照Subagent模式，主Agent需要把Subagents当做一个个Tool来协调使用。所以，我们接下来就先定义3个Tool，分别对应3个Subagent.
# 这些tool读取state中的婚礼信息，然后调用对应的subagent，分别完成自己的任务。
@tool
async def search_flights(runtime: ToolRuntime) -> str:
    """旅行Agent搜索飞往理想婚礼地点的航班."""
    origin = runtime.state["origin"]
    destination = runtime.state["destination"]

    content = f"查询从{origin}到{destination}的航班"
    response = await travel_agent.ainvoke({"messages": [HumanMessage(content)]})
    return response['messages'][-1].content

@tool
def search_venues(runtime: ToolRuntime) -> str:
    """场地Agent根据给定的地点和宾客数量寻找最佳的婚礼场地."""
    destination = runtime.state["destination"]
    capacity = runtime.state["guest_count"]

    content = f"在{destination}寻找适合{capacity}宾客的婚礼场地"
    response = venue_agent.invoke({"messages": [HumanMessage(content)]})
    return response['messages'][-1].content

@tool
def suggest_playlist(runtime: ToolRuntime) -> str:
    """播放列表Agent根据类型风格策划完美的播放列表."""
    genre = runtime.state["genre"]

    content = f"策划符合{genre}风格的婚礼播放列表的曲目"
    response = playlist_agent.invoke({"messages": [HumanMessage(content)]})
    return response['messages'][-1].content

In [ ]:
from langchain.agents import create_agent

coordinator = create_agent(
    model="deepseek-flash",
    tools=[search_flights, search_venues, suggest_playlist, update_state],
    state_schema=WeddingState,
    system_prompt="""
    你是婚礼协调员。你负责与用户沟通，找到更新状态所需的所有信息，并更新到state。
    一旦完成，你就可以委派任务了。将查找航班、搜索场地和策划播放列表的任务委派给您的专家Agent。
    当你收到他们的答案，为我设计一场完美的婚礼方案。
    """
)

response = await coordinator.ainvoke(
    {
        "messages": [HumanMessage(content="我来自北京，我想在上海举办一场100人的婚礼，爵士风格的。")],
    }
)

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

我来自北京，我想在上海举办一场100人的婚礼，爵士风格的。
================================== Ai Message ==================================

太好了！您提供的信息非常完整。让我先更新婚礼状态，然后为您委派专家团队。

- **出发地**：北京
- **目的地**：上海
- **宾客人数**：100人
- **婚礼风格**：爵士
Tool Calls:
  update_state (call_00_kSmhDUkzUkMs3XBAMZrc2847)
 Call ID: call_00_kSmhDUkzUkMs3XBAMZrc2847
  Args:
    origin: 北京
    destination: 上海
    guest_count: 100
    genre: 爵士
================================= Tool Message =================================
Name: update_state

成功更新state
================================== Ai Message ==================================

状态已更新成功！现在让我同时委派专家团队为您处理航班、场地和播放列表的事宜。
Tool Calls:
  search_flights (call_00_IvgMkpoi2Ru9EzTZo84p5762)
 Call ID: call_00_IvgMkpoi2Ru9EzTZo84p5762
  Args:
  search_venues (call_01_JKlaW83126u3Sb477GOb5237)
 Call ID: call_01_JKlaW83126u3Sb477GOb5237
  Args:
  suggest_playlist (call_02_NvDbsxHheQyOMNxB3hDR0716)
 Call ID: call_02_NvDbsxHh

: 